In [1]:
import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
import pandas as pd

large_data_dir = gf_utils.large_data_dir

In [ ]:
all_df = pd.DataFrame()

for lib in ['4-plex','1','2']: ### exclude 1-plex because it only used JAK2 probe
    if lib == '4-plex':
        BCs = ['BC001', 'BC002', 'BC003', 'BC004']
    elif lib == '1-plex':
        BCs = ['BC001']
    else:
        BCs = ['BC001', 'BC002', 'BC003', 'BC004', 'BC005', 'BC006', 'BC007', 'BC008', 'BC009', 'BC010', 'BC011', 'BC012', 'BC013', 'BC014', 'BC015', 'BC016']

    for BC in BCs:
        sample = BC + '_' + lib

        pcr_swap_likelihoods = pd.read_csv('../../4_figure_MPN_metrics/data/swap_probabilities/likelihood_tables_' + lib.replace('-','') + '/patient_' + BC + '_swap_probabilities.csv')
        pcr_swap_likelihoods.rename(columns={'likelihood':'no_pcr_swap_likelihood'}, inplace=True)
        pcr_swap_likelihoods['pcr_swap_likelihood'] = 1 - pcr_swap_likelihoods['no_pcr_swap_likelihood']
        threshold = pcr_swap_likelihoods.loc[pcr_swap_likelihoods['pcr_swap_likelihood'] < 0.1]['pcr_duplicate_count'].min() - 1 ### set threshold 1 below so UMIs at threshold are included; same as high confidence counts elsewhere

        gf_dirs = {}
        ### first get probe_reads to use for the patient
        if lib == '4-plex':
            gf_dirs[0] = large_data_dir + 'gf_MPN/gf_MPN_4plex_1/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
            gf_dirs[1] = large_data_dir + 'gf_MPN/gf_MPN_4plex_2/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        elif lib == '1-plex':
            gf_dirs[0] = large_data_dir + 'gf_MPN/MPN_1plex/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        elif lib == '1': ## 16-plex
            gf_dirs[0] = large_data_dir + 'gf_MPN/gf_MPN_16plex_part' + lib + '_1/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
            gf_dirs[1] = large_data_dir + 'gf_MPN/gf_MPN_16plex_part' + lib + '_2/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        elif lib == '2': ## 16-plex
            gf_dirs[1] = large_data_dir + 'gf_MPN/gf_MPN_16plex_part' + lib + '_2/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        min_percent_supporting = 0.9
        collapse_across_probes = True
        iter = 0
        manifest = gf_utils.get_manifest(gf_dirs[list(gf_dirs.keys())[0]])
        if len(manifest.loc[(manifest['name'] == 'NFE2 c.782_785del') & (manifest['gapfill_from_transcriptome'].notna())]) > 0:
            for key,gf_dir in gf_dirs.items():
                if iter == 0:
                    manifest = gf_utils.get_manifest(gf_dir)
                    probe_reads = gf_utils.get_input_probe_reads(gf_dir, read_threshold = threshold, cell_barcode_suffix = '-' + str(key), min_percent_supporting=min_percent_supporting, collapse_across_probes=collapse_across_probes)
                else:
                    probe_reads = pd.concat([probe_reads, gf_utils.get_input_probe_reads(gf_dir, read_threshold = threshold, cell_barcode_suffix = '-' + str(key), min_percent_supporting=min_percent_supporting, collapse_across_probes=collapse_across_probes)], ignore_index=True)
                iter += 1       

            manifest['name'] = manifest['name'].str.replace('_1$', '', regex=True) ### fix name for dual probes which will end with _0 or _1
            manifest['name'] = manifest['name'].str.replace('_0$', '', regex=True) ### fix name for dual probes which will end with _0 or _1

            probe_reads = probe_reads.merge(manifest[['name']], left_on='probe_idx', right_index=True, how='left')

            df = pd.DataFrame({
                'n_UMIs': probe_reads.loc[probe_reads['name'] == 'NFE2 c.782_785del']['barcode'].value_counts(),
            })
            df.index = df.index + '-' + sample
            
            all_df = pd.concat([all_df, df])


585762 UMIs found
Collapsing UMIs across probes, 585762 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (60) and min percent supporting (0.9), 400247 UMIs remaining (68.33%)
605246 UMIs found
Collapsing UMIs across probes, 605246 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (60) and min percent supporting (0.9), 399621 UMIs remaining (66.03%)
1619364 UMIs found
Collapsing UMIs across probes, 1619364 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (60) and min percent supporting (0.9), 1008732 UMIs remaining (62.29%)
1675486 UMIs found
Collapsing UMIs across probes, 1675486 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (60) and min percent supporting (0.9), 1003228 UMIs remaining (59.88%)
1815432 UMIs found
Collapsing UMIs across probes, 1815432 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (61) and min percent supporting (0.9), 1001998 UMIs remaining (55.19%)
186855

In [3]:
all_df.to_csv('../output/NFE2_782_785del_UMIs_per_cell.csv')

In [ ]:
all_df = pd.DataFrame()

for lib in ['4-plex','1','2']: ### exclude 1-plex because it only used JAK2 probe
    if lib == '4-plex':
        BCs = ['BC001', 'BC002', 'BC003', 'BC004']
    elif lib == '1-plex':
        BCs = ['BC001']
    else:
        BCs = ['BC001', 'BC002', 'BC003', 'BC004', 'BC005', 'BC006', 'BC007', 'BC008', 'BC009', 'BC010', 'BC011', 'BC012', 'BC013', 'BC014', 'BC015', 'BC016']

    for BC in BCs:
        sample = BC + '_' + lib

        pcr_swap_likelihoods = pd.read_csv('../../4_figure_MPN_metrics/data/swap_probabilities/likelihood_tables_' + lib.replace('-','') + '/patient_' + BC + '_swap_probabilities.csv')
        pcr_swap_likelihoods.rename(columns={'likelihood':'no_pcr_swap_likelihood'}, inplace=True)
        pcr_swap_likelihoods['pcr_swap_likelihood'] = 1 - pcr_swap_likelihoods['no_pcr_swap_likelihood']
        threshold = pcr_swap_likelihoods.loc[pcr_swap_likelihoods['pcr_swap_likelihood'] < 0.1]['pcr_duplicate_count'].min() - 1 ### set threshold 1 below so UMIs at threshold are included; same as high confidence counts elsewhere

        gf_dirs = {}
        ### first get probe_reads to use for the patient
        if lib == '4-plex':
            gf_dirs[0] = large_data_dir + 'gf_MPN/gf_MPN_4plex_1/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
            gf_dirs[1] = large_data_dir + 'gf_MPN/gf_MPN_4plex_2/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        elif lib == '1-plex':
            gf_dirs[0] = large_data_dir + 'gf_MPN/MPN_1plex/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        elif lib == '1': ## 16-plex
            gf_dirs[0] = large_data_dir + 'gf_MPN/gf_MPN_16plex_part' + lib + '_1/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
            gf_dirs[1] = large_data_dir + 'gf_MPN/gf_MPN_16plex_part' + lib + '_2/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        elif lib == '2': ## 16-plex
            gf_dirs[1] = large_data_dir + 'gf_MPN/gf_MPN_16plex_part' + lib + '_2/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        min_percent_supporting = 0.9
        collapse_across_probes = True
        iter = 0
        manifest = gf_utils.get_manifest(gf_dirs[list(gf_dirs.keys())[0]])
        if len(manifest.loc[(manifest['name'] == 'SF3B1 c.1874G>A') & (manifest['gapfill_from_transcriptome'].notna())]) > 0:
            for key,gf_dir in gf_dirs.items():
                if iter == 0:
                    manifest = gf_utils.get_manifest(gf_dir)
                    probe_reads = gf_utils.get_input_probe_reads(gf_dir, read_threshold = threshold, cell_barcode_suffix = '-' + str(key), min_percent_supporting=min_percent_supporting, collapse_across_probes=collapse_across_probes)
                else:
                    probe_reads = pd.concat([probe_reads, gf_utils.get_input_probe_reads(gf_dir, read_threshold = threshold, cell_barcode_suffix = '-' + str(key), min_percent_supporting=min_percent_supporting, collapse_across_probes=collapse_across_probes)], ignore_index=True)
                iter += 1       

            manifest['name'] = manifest['name'].str.replace('_1$', '', regex=True) ### fix name for dual probes which will end with _0 or _1
            manifest['name'] = manifest['name'].str.replace('_0$', '', regex=True) ### fix name for dual probes which will end with _0 or _1

            probe_reads = probe_reads.merge(manifest[['name']], left_on='probe_idx', right_index=True, how='left')

            df = pd.DataFrame({
                'n_UMIs': probe_reads.loc[probe_reads['name'] == 'SF3B1 c.1874G>A']['barcode'].value_counts(),
            })
            df.index = df.index + '-' + sample
            
            all_df = pd.concat([all_df, df])

585762 UMIs found
Collapsing UMIs across probes, 585762 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (60) and min percent supporting (0.9), 400247 UMIs remaining (68.33%)
605246 UMIs found
Collapsing UMIs across probes, 605246 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (60) and min percent supporting (0.9), 399621 UMIs remaining (66.03%)
1619364 UMIs found
Collapsing UMIs across probes, 1619364 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (60) and min percent supporting (0.9), 1008732 UMIs remaining (62.29%)
1675486 UMIs found
Collapsing UMIs across probes, 1675486 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (60) and min percent supporting (0.9), 1003228 UMIs remaining (59.88%)
1815432 UMIs found
Collapsing UMIs across probes, 1815432 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (61) and min percent supporting (0.9), 1001998 UMIs remaining (55.19%)
186855

In [5]:
all_df.to_csv('../output/SF3B1_1874_UMIs_per_cell.csv')